In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Import successful!")



Import successful!


In [67]:
matches = pd.read_csv("../data/raw/matches.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")

print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)

print("MATCHES COLUMNS")
print(matches.columns.tolist())

print("\nDELIVERIES COLUMNS")

print(matches.columns.tolist())
print(deliveries.columns.tolist())

matches_raw = matches.copy()
deliveries_raw = deliveries.copy()

print("Backup created successfully!")

print("Duplicate match rows:",
      matches.duplicated().sum())
print("Duplicate delivery rows:",
      deliveries.duplicated().sum())

print("MATCHES MISSING VALUES")
print(matches.isnull().sum())
print("\nDELIVERIES MISSING VALUES")
print(deliveries.isnull().sum())

if "result" in matches.columns:
    print(matches["result"].value_counts(dropna=False))

if "date" in matches.columns:
    matches["date"] = pd.to_datetime(
        matches["date"],
        errors="coerce"
    )
    print(matches["date"].dtype)

if "season" in matches.columns:
    print(sorted(matches["season"].unique()))   
if "season" in matches.columns:
    print("Number of seasons:",
          matches["season"].nunique())  

teams = pd.unique(
    matches[["team1", "team2"]].values.ravel()
)

print("Teams:")
for team in teams:
    print(team)

print("\nNumber of teams:", len(teams)) 

team_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiant": "Rising Pune Supergiants"
}
matches["team1"] = matches["team1"].replace(team_mapping)
matches["team2"] = matches["team2"].replace(team_mapping)
matches["toss_winner"] = matches["toss_winner"].replace(team_mapping)
matches["winner"] = matches["winner"].replace(team_mapping)
deliveries["batting_team"] = deliveries["batting_team"].replace(team_mapping)
deliveries["bowling_team"] = deliveries["bowling_team"].replace(team_mapping)

print("Unique batters:",
      deliveries["batter"].nunique())
print("Unique bowlers:",
      deliveries["bowler"].nunique())
print(deliveries["batter"].unique()[:20])
print(deliveries[
    ["batsman_runs", "extra_runs", "total_runs"]
].head())
match_runs = (
    deliveries
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index()
)

match_runs.columns = [
    "match_id",
    "total_match_runs"
]

print(match_runs.head())

matches = matches.merge(
    match_runs,
    left_on="id",
    right_on="match_id",
    how="left"
)
print(matches.head())

first_innings = (
    deliveries[deliveries["inning"] == 1]
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index()
)

first_innings.columns = [
    "match_id",
    "first_innings_score"
]

print(first_innings.head())
matches = matches.merge(
    first_innings,
    left_on="id",
    right_on="match_id",
    how="left"
)
print(matches.head())

second_innings = (
    deliveries[deliveries["inning"] == 2]
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index()
)

second_innings.columns = [
    "id",
    "second_innings_score"
]

matches = matches.merge(
    second_innings,
    on="id",
    how="left"
)


# 1. Reload clean raw data to reset matches DataFrame in memory
matches = pd.read_csv("../data/raw/matches.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")

# 2. Compute fours aggregated by match_id
fours = (
    deliveries[deliveries["batsman_runs"] == 4]
    .groupby("match_id")
    .size()
    .reset_index(name="fours")
)

sixes =(
    deliveries[deliveries["batsman_runs"] == 6]
    .groupby("match_id")
    .size()
    .reset_index(name="sixes")
)
# 3. Perform the clean merge
matches = matches.merge(
    fours,
    left_on="id",
    right_on="match_id",
    how="left"
)

matches = matches.merge(
    sixes,
    left_on="id",
    right_on="match_id",
    how="left"
)

# 4. Fill NaN values for matches where zero 4s were hit
matches["fours"] = matches["fours"].fillna(0).astype(int)
matches["sixes"] = matches["sixes"].fillna(0).astype(int)

# 5. Clean up redundant column if created
if "match_id" in matches.columns:
    matches.drop(columns=["match_id"], inplace=True)

print(matches[["id", "fours"]].head())
print(matches[["id", "sixes"]].head())

wickets = (
    deliveries[deliveries["is_wicket"] == 1]
    .groupby("match_id")
    .size()
    .reset_index(name="wickets")
)
matches = matches.merge(
    wickets,
    left_on="id",
    right_on="match_id",
    how="left"
)
matches["wickets"] = matches["wickets"].fillna(0)
print(matches[["id","wickets"]].head())

player_stats = (
    deliveries
    .groupby("batter")
    .agg(
        runs=("batsman_runs", "sum"),
        balls=("batsman_runs", "count"),
        sixes=("batsman_runs",
               lambda x: (x == 6).sum()),
        fours=("batsman_runs",
               lambda x: (x == 4).sum())
    )
    .reset_index()
)
player_stats = (
    deliveries
    .groupby("batter")
    .agg(
        runs=("batsman_runs", "sum"),
        balls=("batsman_runs", "count"),
        sixes=("batsman_runs",
               lambda x: (x == 6).sum()),
        fours=("batsman_runs",
               lambda x: (x == 4).sum())
    )
    .reset_index()
)
print(player_stats.head())

player_stats["strike_rate"] = (
    player_stats["runs"] /
    player_stats["balls"]
) * 100
player_stats["strike_rate"] = (
    player_stats["strike_rate"].round(2)
)
print(
    player_stats
    .sort_values("runs", ascending=False)
    .head(10)
)

bowling_stats = (
    deliveries
    .groupby("bowler")
    .agg(
        wickets=("is_wicket", "sum"),
        runs_conceded=("total_runs", "sum"),
        balls=("total_runs", "count")
    )
    .reset_index()
)
print(bowling_stats.head())

bowling_stats["overs"] = (
    bowling_stats["balls"] / 6
).round(2)

bowling_stats["economy"] = (
    bowling_stats["runs_conceded"] /
    bowling_stats["overs"].replace(0, np.nan)
).round(2)
print(
    bowling_stats
    .sort_values("wickets", ascending=False)
    .head(10)
)

player_stats["boundary_runs"] = (
    player_stats["fours"] * 4 +
    player_stats["sixes"] * 6
)
player_stats["boundary_percentage"] = (
    player_stats["boundary_runs"] /
    player_stats["runs"].replace(0, np.nan)
) * 100
player_stats["boundary_percentage"] = (
    player_stats["boundary_percentage"].round(2)
)

powerplay = deliveries[
    deliveries["over"] < 6
]
powerplay_runs = (
    powerplay
    .groupby("batting_team")["total_runs"]
    .sum()
    .sort_values(ascending=False)
)

print(powerplay_runs.head(10))

death_overs = deliveries[
    deliveries["over"] >= 16
]
death_runs = (
    death_overs
    .groupby("batting_team")["total_runs"]
    .sum()
    .sort_values(ascending=False)
)

print(death_runs.head(10))

toss_match_winner = (
    matches["toss_winner"] ==
    matches["winner"]
)
toss_win_percentage = (
    toss_match_winner.mean() * 100
)

print(
    "Toss winner also won:",
    round(toss_win_percentage, 2),
    "%"
)

if "toss_decision" in matches.columns:
    print(
        matches["toss_decision"].value_counts()
    )

team1 = matches[
    ["id", "season", "team1", "winner"]
].copy()

team1.columns = [
    "id",
    "season",
    "team",
    "winner"
]

team2 = matches[
    ["id", "season", "team2", "winner"]
].copy()

team2.columns = [
    "id",
    "season",
    "team",
    "winner"
]

team_matches = pd.concat(
    [team1, team2],
    ignore_index=True
) 
matches_played = (
    team_matches
    .groupby("team")
    .size()
    .reset_index(name="matches_played")
)
wins = (
    team_matches[
        team_matches["team"] ==
        team_matches["winner"]
    ]
    .groupby("team")
    .size()
    .reset_index(name="wins")
)
team_stats = matches_played.merge(
    wins,
    on="team",
    how="left"
)
team_stats = matches_played.merge(
    wins,
    on="team",
    how="left"
) 

team_stats["win_percentage"] = (
    team_stats["wins"] /
    team_stats["matches_played"]
) * 100
team_stats["win_percentage"] = (
    team_stats["win_percentage"].round(2)
)
print(
    team_stats
    .sort_values(
        "win_percentage",
        ascending=False
    )
)
# 1. Reload raw data
matches = pd.read_csv("../data/raw/matches.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")

# 2. Calculate match runs & first innings scores from deliveries
match_runs = (
    deliveries
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index(name="total_runs")
)

first_innings = (
    deliveries[deliveries["inning"] == 1]
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index(name="first_innings_score")
)

# 3. Merge both aggregated datasets into matches
matches = matches.merge(match_runs, left_on="id", right_on="match_id", how="left")
matches = matches.merge(first_innings, left_on="id", right_on="match_id", how="left")

# Drop redundant match_id columns created by merges
matches.drop(columns=[col for col in matches.columns if col.startswith("match_id")], inplace=True)

season_stats = (
    matches
    .groupby("season")
    .agg(
        total_matches=("id", "count"),
        total_runs=("total_runs", "sum"),
        avg_first_innings=(
            "first_innings_score",
            "mean"
        )
    )
    .reset_index()
)
season_stats["avg_first_innings"] = (
    season_stats["avg_first_innings"].round(2)
)
print(season_stats)

print(matches.shape)
print(matches.head())

print(player_stats.shape)
print(player_stats.head())

print(bowling_stats.shape)
print(bowling_stats.head())

print(team_stats.shape)
print(team_stats.head())

print(season_stats.shape)
print(season_stats.head())

matches.to_csv(
    "../data/processed/matches_cleaned.csv",
    index=False
)

deliveries.to_csv(
    "../data/processed/deliveries_cleaned.csv",
    index=False
)

player_stats.to_csv(
    "../data/processed/player_stats.csv",
    index=False
)

team_stats.to_csv(
    "../data/processed/team_stats.csv",
    index=False
)

bowling_stats.to_csv(
    "../data/processed/bowling_stats.csv",
    index=False
)

season_stats.to_csv(
    "../data/processed/season_stats.csv",
    index=False
)

Matches: (1095, 20)
Deliveries: (260920, 17)
MATCHES COLUMNS
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

DELIVERIES COLUMNS
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']
Backup created successfully!
Duplicate match rows: 0
Duplicate delivery rows: 0
MATCHES MISSING VALUES
id                    0
season                0
city                 51
date                  0
match_type     

In [68]:
print("===================================")
print("DAY 2 DATA CLEANING COMPLETED")
print("===================================")

print("\nMatches:", matches.shape)
print("Deliveries:", deliveries.shape)
print("Player Stats:", player_stats.shape)
print("Bowling Stats:", bowling_stats.shape)
print("Team Stats:", team_stats.shape)
print("Season Stats:", season_stats.shape)

print("\nFiles created:")
print("✓ matches_cleaned.csv")
print("✓ deliveries_cleaned.csv")
print("✓ player_stats.csv")
print("✓ bowling_stats.csv")
print("✓ team_stats.csv")
print("✓ season_stats.csv")

DAY 2 DATA CLEANING COMPLETED

Matches: (1095, 22)
Deliveries: (260920, 17)
Player Stats: (673, 8)
Bowling Stats: (530, 6)
Team Stats: (19, 4)
Season Stats: (17, 4)

Files created:
✓ matches_cleaned.csv
✓ deliveries_cleaned.csv
✓ player_stats.csv
✓ bowling_stats.csv
✓ team_stats.csv
✓ season_stats.csv
